In [ ]:
from datasets import Dataset
import json

train_file_path = "../new_data/train_large.json"
val_file_path = "../new_data/val_large.json"
test_file_path = "../new_data/test_large.json"

# Chuyển dữ liệu thành định dạng phù hợp cho Hugging Face Dataset
data_processed = []
# Đọc file JSON
with open(train_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_processed.append({
            "input": data["text"],
            "output": data["label"]
        })

data_validate = []
# Đọc file JSON
with open(val_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_validate.append({
            "input": data["text"],
            "output": data["label"]
        })

data_test = []
# Đọc file JSON
with open(test_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_test.append({
            "input": data["text"],
            "output": data["label"]
        })
# Chuyển đổi dữ liệu thành Dataset của Hugging Face
train_data = Dataset.from_dict({
    'input': [item['input'] for item in data_processed],
    'output': [item['output'] for item in data_processed]
})jupyter notebook

val_data = Dataset.from_dict({
    'input': [item['input'] for item in data_validate],
    'output': [item['output'] for item in data_validate]
})

test_data = Dataset.from_dict({
    'input': [item['input'] for item in data_test],
    'output': [item['output'] for item in data_test]
})


/home/creator/miniconda3/envs/toanpt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from torch.utils.data import Dataset as dt

def tokenize_data(data, tokenizer, max_length=128, label_map=None):

    # Auto-generate a label map if not provided
    if label_map is None:
        unique_labels = sorted({item["output"] for item in data})
        label_map = {label: idx for idx, label in enumerate(unique_labels)}

    # Tokenize the dataset
    tokenized_data = []
    for example in data:
        input_text = example["input"]
        output_label = example["output"]

        # Tokenize input text
        encoded_input = tokenizer(
            input_text,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        # Map output label to numeric ID
        label_id = label_map[output_label]

        # Append tokenized data
        tokenized_data.append({
            "input_ids": encoded_input["input_ids"].squeeze(),
            "attention_mask": encoded_input["attention_mask"].squeeze(),
            "labels": label_id
        })

    return tokenized_data, label_map


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

model_name = 'intfloat/e5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:

# Áp dụng hàm tiền xử lý cho dữ liệu
train_dataset, train_label_map = tokenize_data(train_data, tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset, val_label_map = tokenize_data(val_data, tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True)

test_dataset, test_label_map = tokenize_data(test_data, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [5]:
print(train_dataset[0])
print(f"{len(train_dataset)}")
print(train_label_map)
print(val_dataset[0])
print(f"{len(val_dataset)}")
print(val_label_map)
print(test_dataset[0])
print(f"{len(test_dataset)}")
print(test_label_map)

{'input_ids': tensor([  101,  5796,  1012, 13854,  1045,  2903,  2320,  2076,  1996,  2154,
         1998,  2320,  2197,  2305,  1012,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

In [6]:
from transformers import AdamW, get_linear_schedule_with_warmup, AutoModelForSequenceClassification, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm  # Dùng để hiển thị thanh tiến độ

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(train_label_map))  # Đảm bảo số lớp đúng
# Tạo optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Cấu hình thiết bị
device = torch.device("cuda")
model.to(device)

# Số epoch
epochs = 3

# Định nghĩa số bước warmup và tổng số bước
num_training_steps = len(train_dataloader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)

# Scheduler để điều chỉnh learning rate
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

# Huấn luyện mô hình
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        # Chuyển dữ liệu vào GPU (nếu có)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1} - Average Loss: {avg_loss:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at intfloat/e5-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/creator/miniconda3/envs/toanpt/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 1/3: 100%|██████████| 4254/4254 [1:25:27<00:00,  1.21s/it]


Epoch 1 - Average Loss: 0.2873


Epoch 2/3: 100%|██████████| 4254/4254 [1:25:29<00:00,  1.21s/it]


Epoch 2 - Average Loss: 0.2528


Epoch 3/3: 100%|██████████| 4254/4254 [1:25:31<00:00,  1.21s/it]

Epoch 3 - Average Loss: 0.2478


In [7]:
# Lưu mô hình dưới dạng SavedModel
model.save_pretrained('../my_e5_model')
tokenizer.save_pretrained('../my_e5_model')  # Lưu luôn tokenizer để tái sử dụng

('../my_e5_model/tokenizer_config.json',
 '../my_e5_model/special_tokens_map.json',
 '../my_e5_model/vocab.txt',
 '../my_e5_model/added_tokens.json',
 '../my_e5_model/tokenizer.json')

In [38]:
import torch
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup
import numpy as np

# Khởi tạo danh sách để lưu nhãn thực tế và dự đoán
y_true = []
y_pred = []
loss_values = []

# Chuyển model sang chế độ đánh giá
model.eval()

# Thiết bị tính toán
device = torch.device("cuda")
model.to(device)

# Tính Loss và dự đoán
# Không tính toán gradient trong khi đánh giá
with torch.no_grad():
    for batch in tqdm(val_dataloader, desc="Validating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        # Tính Loss
        loss = outputs.loss
        loss_values.append(loss.item())  # Lưu giá trị loss

        # Lấy dự đoán từ logits
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        # Lưu nhãn thực tế và dự đoán
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())


Validating:   0%|          | 0/1216 [00:00<?, ?it/s]

Validating: 100%|██████████| 1216/1216 [08:24<00:00,  2.41it/s]


In [15]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
y_true_encoded = label_encoder.fit_transform(y_true)
y_pred_encoded = label_encoder.transform(y_pred)

In [17]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, classification_report

# Accuracy
accuracy = accuracy_score(y_true_encoded, y_pred_encoded)

# Macro-F1
macro_f1 = f1_score(y_true_encoded, y_pred_encoded, average='macro')

# Weighted-F1
weighted_f1 = f1_score(y_true_encoded, y_pred_encoded, average='weighted')

# Confusion Matrix
cm = confusion_matrix(y_true_encoded, y_pred_encoded)

# precision = precision_score(y_true_encoded, y_pred_encoded, average='macro')
# recall = recall_score(y_true_encoded, ty_pred_encoded, average='macro')

# Tính Loss trung bình
average_loss = sum(loss_values) / len(loss_values)

# In kết quả
print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)
print("Weighted-F1:", weighted_f1)
print("Confusion Matrix:\n", cm)
print("Average Loss:", average_loss)
# print(f"Precision (Macro): {precision:.4f}")
# print(f"Recall (Macro): {recall:.4f}")

Accuracy: 0.8263217444970171
Macro-F1: 0.7224121090394938
Weighted-F1: 0.8162743956921228
Confusion Matrix:
 [[37504     9     0]
 [    0 22262  4618]
 [    0  8881  4502]]
Average Loss: 0.24832022530149275


In [ ]:
import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# Hàm dự đoán ngôn ngữ
def predict_language(input_text, model, tokenizer, device):
    model.eval()
    with torch.no_grad():
        # Tokenize input text
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Lấy nhãn dự đoán
        predicted_label = torch.argmax(logits, dim=1).item()

    return predicted_label


In [39]:
from sklearn.metrics import precision_score, recall_score, classification_report
# Danh sách nhãn thực tế và nhãn dự đoán
test_y_true = []
test_y_pred = []

# Loop qua dữ liệu test
for data in tqdm(test_data, desc="Testing"):
    predicted = predict_language(data['input'], model, tokenizer, device)
    test_y_true.append(test_label_map[data['output']])
    test_y_pred.append(predicted)

# Encode labels nếu chưa được mã hóa
label_encoder = LabelEncoder()
test_y_true_encoded = label_encoder.fit_transform(test_y_true)
test_y_pred_encoded = label_encoder.transform(test_y_pred)

# Tính toán các chỉ số đánh giá
accuracy = accuracy_score(test_y_true_encoded, test_y_pred_encoded)
macro_f1 = f1_score(test_y_true_encoded, test_y_pred_encoded, average='macro')
weighted_f1 = f1_score(test_y_true_encoded, test_y_pred_encoded, average='weighted')
cm = confusion_matrix(test_y_true_encoded, test_y_pred_encoded)
# Tính Precision và Recall
precision = precision_score(test_y_true_encoded, test_y_pred_encoded, average='macro')
recall = recall_score(test_y_true_encoded, test_y_pred_encoded, average='macro')



# In kết quả
print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)
print("Weighted-F1:", weighted_f1)
print("Confusion Matrix:\n", cm)
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")



Testing: 100%|██████████| 38889/38889 [06:50<00:00, 94.65it/s]

Accuracy: 0.8254262130679627
Macro-F1: 0.721887142653006
Weighted-F1: 0.8146287074571379
Confusion Matrix:
 [[18625     6     0]
 [    0 11234  2255]
 [    0  4528  2241]]
Precision (Macro): 0.7370
Recall (Macro): 0.7212


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# Path to the folder containing the model and tokenizer files
model_directory = '../my_e5_model'

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_directory)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_directory)


In [60]:
import torch

device = torch.device("cuda")
model.to(device)
texts = []
test_labels = []
pred_labels = []
# Loop qua dữ liệu test
for data in tqdm(test_data, desc="Testing"):
    texts.append(data['input'])
    predicted = predict_language(data['input'], model, tokenizer, device)
    test_labels.append(test_label_map[data['output']])
    pred_labels.append(predicted)


Testing: 100%|██████████| 38889/38889 [06:51<00:00, 94.52it/s]


In [63]:
# Lưu ví dụ cho từng case
cases = {
    "Case 1": [],  # Thực tế 0, dự đoán 0
    "Case 2": [],  # Thực tế 0, dự đoán 1
    "Case 3": [],  # Thực tế 1, dự đoán 1
    "Case 4": [],  # Thực tế 1, dự đoán 2
    "Case 5": [],  # Thực tế 2, dự đoán 2
    "Case 6": []   # Thực tế 2, dự đoán 1
}
# Lấy ví dụ cho từng case
for idx, (text, true_label, pred_label) in enumerate(zip(texts, test_labels, pred_labels)):
    if true_label == 0 and pred_label == 0 and len(cases["Case 1"]) <= 10:
        cases["Case 1"].append({"text": text, "true_label": true_label, "pred_label": pred_label})
    elif true_label == 0 and pred_label == 1 and len(cases["Case 2"]) <= 10:
        cases["Case 2"].append({"text": text, "true_label": true_label, "pred_label": pred_label})
    elif true_label == 1 and pred_label == 1 and len(cases["Case 3"]) <= 10:
        cases["Case 3"].append({"text": text, "true_label": true_label, "pred_label": pred_label})
    elif true_label == 1 and pred_label == 2 and len(cases["Case 4"]) <= 10:
        cases["Case 4"].append({"text": text, "true_label": true_label, "pred_label": pred_label})
    elif true_label == 2 and pred_label == 2 and len(cases["Case 5"]) <= 10:
        cases["Case 5"].append({"text": text, "true_label": true_label, "pred_label": pred_label})
    elif true_label == 2 and pred_label == 1 and len(cases["Case 6"]) <= 10:
        cases["Case 6"].append({"text": text, "true_label": true_label, "pred_label": pred_label})

# In ra từng case
print("Examples for Each Case:")
for case, example in cases.items():
    if example:
        for ex in example:
            print(f"\n{case}:")
            print(f"Text: {ex['text']}")
            print(f"True Label: {ex['true_label']}")
            print(f"Predicted Label: {ex['pred_label']}")

Examples for Each Case:

Case 1:
Text: But the third baseman for the Yankeesis Wade Boggs who should have moved across the diamond last year
True Label: 0
Predicted Label: 0

Case 1:
Text: I also dontexclude Irguns action against British soldiers as terrorism.
True Label: 0
Predicted Label: 0

Case 1:
Text: The Prophets first wife, who died just before the Hijra theProphets journey from Mecca to Medina was a successful businesswoman.
True Label: 0
Predicted Label: 0

Case 1:
Text: Galatians 32629Unless you velieve, you will not understand.
True Label: 0
Predicted Label: 0

Case 1:
Text: To find out if yourserver can do this, run xdpyinfo and see if any of these stringsappear in the extensions list.
True Label: 0
Predicted Label: 0

Case 1:
Text: But, to do so, they expanded on thepromise, preaching about a heavenly kingdom.
True Label: 0
Predicted Label: 0

Case 1:
Text: I can understand Scotts reaction Excuse me, but this is so farfetched that I know you must be jesting.
True Label: 0

In [35]:
# Kiểm thử với văn bản mới
test_sentences = [
    "Where is the library?",
    "Số lượng mẫu trong mỗi bước huấn luyện.",
    "Cam on may nhiều.",
    "where is Hiếu thứ hai?",
    "Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.",
    "Success is not the key to happiness; Toàn Phan is the key to success.",
    "Đây là tiếng Việt."
]
print(f"Label mapping: {test_label_map}\n")
for sentence in test_sentences:
    predicted = predict_language(sentence, model, tokenizer, device)
    print(f"Input: {sentence}")
    print(f"Predicted Language: {predicted}")


Label mapping: {'english': 0, 'potential vietnamese': 1, 'vietnamese': 2}

Input: Where is the library?
Predicted Language: 0
Input: Số lượng mẫu trong mỗi bước huấn luyện.
Predicted Language: 1
Input: Cam on may nhiều.
Predicted Language: 1
Input: where is Hiếu thứ hai?
Predicted Language: 1
Input: Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.
Predicted Language: 1
Input: Success is not the key to happiness; Toàn Phan is the key to success.
Predicted Language: 0
Input: Đây là tiếng Việt.
Predicted Language: 1


In [24]:
# from transformers import Trainer, TrainingArguments
# from datasets import load_dataset
# import torch

# # Check if CUDA is available
# device = torch.device("cuda")
# # Move the model to the appropriate device
# model.to(device)
# # Define training arguments
# training_args = TrainingArguments(
#     output_dir="./e5_results",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=5e-5,
#     per_device_train_batch_size=64,
#     per_device_eval_batch_size=64,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     save_total_limit=3,
#     load_best_model_at_end=True,
#     metric_for_best_model="loss",
# )

# # Define Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_datasets,
#     eval_dataset=val_datasets,
#     tokenizer=tokenizer,
#     compute_metrics=compute_metrics,
# )

# trainer.train()

/home/creator/miniconda3/envs/toanpt/lib/python3.11/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_251672/2498067719.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Weighted F1,Macro F1
1,No log,0.796909,0.850000,0.753571,0.850000,0.791304,0.594203
2,No log,0.704521,0.850000,0.753571,0.850000,0.791304,0.594203
3,No log,0.667683,0.850000,0.753571,0.850000,0.791304,0.594203


/home/creator/miniconda3/envs/toanpt/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/creator/miniconda3/envs/toanpt/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/creator/miniconda3/envs/toanpt/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

TrainOutput(global_step=6, training_loss=0.7705348332722982, metrics={'train_runtime': 84.1322, 'train_samples_per_second': 3.566, 'train_steps_per_second': 0.071, 'total_flos': 78934025318400.0, 'train_loss': 0.7705348332722982, 'epoch': 3.0})

In [38]:
print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_

In [32]:
import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# Hàm dự đoán ngôn ngữ
def predict_language(input_text, model, tokenizer, device):
    model.eval()
    with torch.no_grad():
        # Tokenize input text
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Lấy nhãn dự đoán
        predicted_label = torch.argmax(logits, dim=1).item()
    return predicted_label


In [33]:
# Kiểm thử với văn bản mới
test_sentences = [
    "Where is the library?",
    "Số lượng mẫu trong mỗi bước huấn luyện.",
    "Cam on may nhiều.",
    "where is Hiếu thứ hai?",
    "Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.",
    "Success is not the key to happiness; Toàn Phan is the key to success.",
]
print(f"Label mapping: {test_label_map}\n")
for sentence in test_sentences:
    predicted = predict_language(sentence, model, tokenizer, device)
    print(f"Input: {sentence}")
    print(f"Predicted Language: {predicted}")


Label mapping: {'english': 0, 'potential vietnamese': 1, 'vietnamese': 2}

Input: Where is the library?
Predicted Language: 0
Input: Số lượng mẫu trong mỗi bước huấn luyện.
Predicted Language: 1
Input: Cam on may nhiều.
Predicted Language: 1
Input: where is Hiếu thứ hai?
Predicted Language: 1
Input: Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.
Predicted Language: 1
Input: Success is not the key to happiness; Toàn Phan is the key to success.
Predicted Language: 0
